# 🚀 Predict Customer Churn — Playground Series S6E3

> **Goal:** একজন customer চলে যাবে কিনা সেটা predict করা (`Churn` = Yes/No)  
> **Metric:** ROC-AUC Score  
> **My Notebook** — step by step করা হয়েছে

---

## 📋 Table of Contents
1. [Libraries Import](#step1)
2. [Data Loading](#step2)
3. [Exploratory Data Analysis (EDA)](#step3)
4. [Data Preprocessing](#step4)
5. [Model Training](#step5)
6. [Model Evaluation](#step6)
7. [Submission তৈরি](#step7)

---
## Step 1: Libraries Import 📦 <a id='step1'></a>

In [ ]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Preprocessing
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score

# Models
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

# Evaluation
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

print("✅ সব libraries import হয়ে গেছে!")

---
## Step 2: Data Loading 📂 <a id='step2'></a>

In [ ]:
# Dataset এর path (তোমার local machine এর path)
TRAIN_PATH = "../Dataset/train.csv"
TEST_PATH  = "../Dataset/test.csv"

# Load করো
train = pd.read_csv(TRAIN_PATH)
test  = pd.read_csv(TEST_PATH)

print(f"Train shape: {train.shape}")
print(f"Test shape : {test.shape}")
print("\n✅ Data load হয়ে গেছে!")

In [ ]:
# প্রথম ৫টা row দেখো
train.head()

In [ ]:
# Column এর নাম ও data type দেখো
train.info()

In [ ]:
# Basic statistics
train.describe()

---
## Step 3: Exploratory Data Analysis (EDA) 🔍 <a id='step3'></a>

> **EDA মানে:** Data কে ভালো করে বোঝা — কোন column এ কী আছে, pattern আছে কিনা দেখা।

In [ ]:
# ── ৩.১ Missing Values চেক করো ──
missing = train.isnull().sum()
missing_pct = (missing / len(train) * 100).round(2)

missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
missing_df = missing_df[missing_df['Missing Count'] > 0]

if missing_df.empty:
    print("✅ কোনো missing value নেই!")
else:
    print(missing_df)

In [ ]:
# ── ৩.২ Target Variable (Churn) এর distribution দেখো ──
churn_counts = train['Churn'].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Churn Distribution', fontsize=16, fontweight='bold')

# Bar chart
axes[0].bar(churn_counts.index, churn_counts.values, color=['#7B2CBF', '#C77DFF'])
axes[0].set_title('Count')
axes[0].set_xlabel('Churn')
axes[0].set_ylabel('Count')
for i, v in enumerate(churn_counts.values):
    axes[0].text(i, v + 100, str(v), ha='center', fontweight='bold')

# Pie chart
axes[1].pie(churn_counts.values, labels=churn_counts.index,
            autopct='%1.1f%%', colors=['#7B2CBF', '#C77DFF'], startangle=90)
axes[1].set_title('Percentage')

plt.tight_layout()
plt.show()

print(churn_counts)

In [ ]:
# ── ৩.৩ Numerical Features এর distribution ──
num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Numerical Features Distribution', fontsize=16, fontweight='bold')

for i, col in enumerate(num_cols):
    # TotalCharges string হতে পারে, convert করো
    data = pd.to_numeric(train[col], errors='coerce').dropna()
    axes[i].hist(data, bins=30, color='#9D4EDD', edgecolor='white')
    axes[i].set_title(col)
    axes[i].set_xlabel('Value')
    axes[i].set_ylabel('Frequency')

plt.tight_layout()
plt.show()

In [ ]:
# ── ৩.৪ Categorical Features vs Churn ──
cat_cols = ['gender', 'SeniorCitizen', 'Partner', 'Dependents',
            'PhoneService', 'Contract', 'PaymentMethod', 'InternetService']

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes = axes.flatten()
fig.suptitle('Categorical Features vs Churn', fontsize=16, fontweight='bold')

for i, col in enumerate(cat_cols):
    churn_by_col = train.groupby([col, 'Churn']).size().unstack(fill_value=0)
    churn_by_col.plot(kind='bar', ax=axes[i], color=['#7B2CBF', '#C77DFF'],
                      edgecolor='white', rot=30)
    axes[i].set_title(col)
    axes[i].set_xlabel('')
    axes[i].legend(title='Churn', loc='upper right')

plt.tight_layout()
plt.show()

In [ ]:
# ── ৩.৫ Correlation Heatmap ──
# Numeric columns নাও
train_temp = train.copy()
train_temp['TotalCharges'] = pd.to_numeric(train_temp['TotalCharges'], errors='coerce')
train_temp['Churn_num'] = (train_temp['Churn'] == 'Yes').astype(int)

numeric_cols = ['tenure', 'MonthlyCharges', 'TotalCharges', 'SeniorCitizen', 'Churn_num']

plt.figure(figsize=(8, 6))
corr = train_temp[numeric_cols].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdPu', linewidths=0.5)
plt.title('Correlation Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Step 4: Data Preprocessing 🛠️ <a id='step4'></a>

> **Preprocessing মানে:** Data কে model এর জন্য উপযুক্ত করে তোলা।
> - Categorical → Numbers
> - Missing values fix
> - Scaling

In [ ]:
# ── ৪.১ Train ও Test একসাথে process করো (সুবিধার জন্য) ──
df_train = train.copy()
df_test  = test.copy()

# Target আলাদা করো
y = (df_train['Churn'] == 'Yes').astype(int)
df_train = df_train.drop(columns=['Churn'])

# id column আলাদা করো (submission এর জন্য)
test_ids = df_test['id']

# id column drop করো training থেকে
df_train = df_train.drop(columns=['id'])
df_test  = df_test.drop(columns=['id'])

print(f"Train X shape: {df_train.shape}")
print(f"Test X shape : {df_test.shape}")
print(f"Target (y) shape: {y.shape}")
print(f"✅ Target encode: No=0, Yes=1 | Churn rate: {y.mean()*100:.2f}%")

In [ ]:
# ── ৪.২ TotalCharges কে numeric এ convert করো ──
df_train['TotalCharges'] = pd.to_numeric(df_train['TotalCharges'], errors='coerce')
df_test['TotalCharges']  = pd.to_numeric(df_test['TotalCharges'],  errors='coerce')

# Missing values (coerce এর কারণে NaN হয়েছে) → median দিয়ে fill করো
median_val = df_train['TotalCharges'].median()
df_train['TotalCharges'].fillna(median_val, inplace=True)
df_test['TotalCharges'].fillna(median_val, inplace=True)

print("✅ TotalCharges fix হয়ে গেছে!")
print(f"   Median value used: {median_val:.2f}")

In [ ]:
# ── ৪.৩ Categorical Columns → Label Encoding ──
# কোন columns গুলো categorical সেগুলো find করো
cat_cols = df_train.select_dtypes(include='object').columns.tolist()
print(f"Categorical columns: {cat_cols}")

le = LabelEncoder()

for col in cat_cols:
    # Train ও Test একসাথে fit করো (unseen labels এর সমস্যা এড়াতে)
    combined = pd.concat([df_train[col], df_test[col]], axis=0)
    le.fit(combined)
    df_train[col] = le.transform(df_train[col])
    df_test[col]  = le.transform(df_test[col])

print("\n✅ Label Encoding সম্পন্ন!")
df_train.head()

In [ ]:
# ── ৪.৪ Feature Engineering (নতুন feature তৈরি) ──

# Average Monthly Charge (tenure এর ভিত্তিতে)
df_train['AvgMonthlyCharge'] = df_train['TotalCharges'] / (df_train['tenure'] + 1)
df_test['AvgMonthlyCharge']  = df_test['TotalCharges']  / (df_test['tenure'] + 1)

print("✅ Feature Engineering সম্পন্ন!")
print(f"Total features: {df_train.shape[1]}")

In [ ]:
# ── ৪.৫ Train/Validation Split ──
X_train, X_val, y_train, y_val = train_test_split(
    df_train, y,
    test_size=0.2,        # 20% validation এ রাখবো
    random_state=42,      # reproducibility এর জন্য
    stratify=y            # Churn ratio maintain করতে
)

print(f"X_train shape: {X_train.shape}")
print(f"X_val   shape: {X_val.shape}")
print(f"y_train Churn rate: {y_train.mean()*100:.2f}%")
print(f"y_val   Churn rate: {y_val.mean()*100:.2f}%")
print("\n✅ Train/Validation split সম্পন্ন!")

---
## Step 5: Model Training 🤖 <a id='step5'></a>

> তিনটা model train করবো এবং তুলনা করবো।

In [ ]:
# ── ৫.১ Logistic Regression (Baseline) ──
print("Training Logistic Regression...")

# Logistic Regression এর জন্য scaling দরকার
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled   = scaler.transform(X_val)

lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train_scaled, y_train)

lr_pred = lr_model.predict_proba(X_val_scaled)[:, 1]
lr_auc = roc_auc_score(y_val, lr_pred)

print(f"✅ Logistic Regression ROC-AUC: {lr_auc:.5f}")

In [ ]:
# ── ৫.২ Random Forest ──
print("Training Random Forest...")

rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=8,
    random_state=42,
    n_jobs=-1          # সব CPU cores use করো speed এর জন্য
)
rf_model.fit(X_train, y_train)

rf_pred = rf_model.predict_proba(X_val)[:, 1]
rf_auc = roc_auc_score(y_val, rf_pred)

print(f"✅ Random Forest ROC-AUC: {rf_auc:.5f}")

In [ ]:
# ── ৫.৩ XGBoost ──
print("Training XGBoost...")

xgb_model = XGBClassifier(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='auc',
    random_state=42,
    n_jobs=-1,
    verbosity=0
)
xgb_model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=100
)

xgb_pred = xgb_model.predict_proba(X_val)[:, 1]
xgb_auc = roc_auc_score(y_val, xgb_pred)

print(f"\n✅ XGBoost ROC-AUC: {xgb_auc:.5f}")

---
## Step 6: Model Evaluation 📊 <a id='step6'></a>

In [ ]:
# ── ৬.১ Model তুলনা ──
results = {
    'Logistic Regression': lr_auc,
    'Random Forest':       rf_auc,
    'XGBoost':             xgb_auc,
}

print("="*45)
print(f"  {'Model':<25} {'ROC-AUC':>10}")
print("="*45)
for name, score in sorted(results.items(), key=lambda x: x[1], reverse=True):
    print(f"  {name:<25} {score:>10.5f}")
print("="*45)

best_model_name = max(results, key=results.get)
print(f"\n🏆 Best Model: {best_model_name} ({results[best_model_name]:.5f})")

In [ ]:
# ── ৬.২ Best Model এর Confusion Matrix ──

# XGBoost কে use করছি (সাধারণত সেরা হয়)
best_pred_label = (xgb_pred >= 0.5).astype(int)
cm = confusion_matrix(y_val, best_pred_label)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion Matrix
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['No Churn', 'Churn'])
disp.plot(ax=axes[0], colorbar=False, cmap='RdPu')
axes[0].set_title('Confusion Matrix (XGBoost)', fontsize=13, fontweight='bold')

# Feature Importance
fi = pd.Series(xgb_model.feature_importances_, index=df_train.columns)
fi = fi.sort_values(ascending=True).tail(15)
fi.plot(kind='barh', ax=axes[1], color='#9D4EDD')
axes[1].set_title('Top 15 Feature Importances (XGBoost)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Importance')

plt.tight_layout()
plt.show()

print("\nClassification Report:")
print(classification_report(y_val, best_pred_label, target_names=['No Churn', 'Churn']))

---
## Step 7: Submission তৈরি করো 🎯 <a id='step7'></a>

In [ ]:
# ── ৭.১ Test data তে prediction করো ──
test_pred = xgb_model.predict_proba(df_test)[:, 1]

# Submission file তৈরি করো
submission = pd.DataFrame({
    'id':    test_ids,
    'Churn': test_pred
})

submission.to_csv('submission.csv', index=False)

print("✅ submission.csv তৈরি হয়ে গেছে!")
print(f"\nShape: {submission.shape}")
submission.head(10)

In [ ]:
# ── ৭.২ Prediction Distribution দেখো ──
plt.figure(figsize=(8, 4))
plt.hist(test_pred, bins=50, color='#7B2CBF', edgecolor='white')
plt.title('Test Prediction Distribution', fontsize=14, fontweight='bold')
plt.xlabel('Predicted Churn Probability')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

print(f"Mean predicted churn probability: {test_pred.mean()*100:.2f}%")

---
## 🎉 Done! 

### Summary:
| Step | কী করলাম |
|------|----------|
| 1 | Libraries import করলাম |
| 2 | train.csv এবং test.csv লোড করলাম |
| 3 | EDA — Data ভালো করে বুঝলাম |
| 4 | Preprocessing — Data কে model উপযোগী করলাম |
| 5 | 3টা model train করলাম (LR, RF, XGB) |
| 6 | ROC-AUC দিয়ে evaluate করলাম |
| 7 | submission.csv তৈরি করে Kaggle এ submit করতে পারবো |

### ⏭️ পরবর্তী উন্নতির সুযোগ:
- LightGBM বা CatBoost try করো
- Cross-Validation (K-Fold) দিয়ে আরো robust result পাও
- Hyperparameter tuning করো (Optuna)
- Ensemble করো (multiple models এর average)
- আরো Feature Engineering করো